<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/%5BBDA%5D_2%E1%84%8C%E1%85%AE%E1%84%8E%E1%85%A1_%E1%84%90%E1%85%A6%E1%86%A8%E1%84%89%E1%85%B3%E1%84%90%E1%85%B3_%E1%84%8C%E1%85%A5%E1%86%AB%E1%84%8E%E1%85%A5%E1%84%85%E1%85%B5_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8_ipynb%EC%9D%98_%EC%8B%A4%EC%8A%B5%20%EB%B0%8F%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 2주차 | 텍스트 전처리 (Text Preprocessing)

**전처리란?**  
텍스트 데이터를 분석하기 전에 **깨끗하게 다듬는 작업**입니다.  
요리 전에 재료를 씻고 손질하는 것과 똑같아요! 🍳

---

**📌 사용법**
- 셀을 클릭하고 **`Shift + Enter`** 를 누르면 실행됩니다
- **위에서 아래로 순서대로** 실행해주세요!
- `[ ]` 가 `[숫자]` 로 바뀌면 실행 완료

---

| 챕터 | 주제 |
|------|------|
| CH 0 | 환경 설정 |
| CH 1 | 왜 전처리가 필요할까? |
| CH 2 | 전처리 4단계 흐름 |
| CH 3 | 토큰화 |
| CH 4 | 정제 (노이즈 제거) |
| CH 5 | 정규화 (표현 통일) |
| CH 6 | 불용어 처리 |
| CH 7 | 한국어 형태소 분석 |
| CH 8 | 파이프라인 통합 실습 |
| CH 9 | 도전 과제 🏆 |

---
## ⚙️ CH 0. 환경 설정

> **가장 먼저 실행해주세요!** 이후 모든 실습에 필요한 도구를 설치합니다.  
> 설치에 약 1~2분 정도 걸립니다. ☕

In [1]:
# KoNLPy는 내부적으로 Java를 사용합니다
# Java를 먼저 설치해야 KoNLPy가 작동합니다
!apt-get install default-jdk -y

# konlpy : 한국어 형태소 분석 라이브러리
# nltk   : 영어 자연어 처리 라이브러리
# --quiet : 설치 메시지를 간략하게 출력
!pip install konlpy nltk --quiet

# 설치 완료 메시지
print("✅ 설치 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jdk default-jdk-headless default-jre
  default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jn

In [3]:
# ── 라이브러리 불러오기 ──────────────────────

import re                    # 정규표현식 : 텍스트에서 특정 패턴을 찾거나 바꿀 때 사용
from collections import Counter  # Counter : 단어 빈도를 쉽게 셀 때 사용

import nltk                             # 영어 자연어 처리 도구 모음
nltk.download('punkt',     quiet=True)  # 문장/단어 토큰화 모델
nltk.download('punkt_tab', quiet=True)  # 토큰화 추가 데이터
nltk.download('stopwords', quiet=True)  # 영어 불용어 목록
nltk.download('wordnet',   quiet=True)  # 영어 단어 원형 데이터

from konlpy.tag import Okt   # Okt : KoNLPy에서 가장 많이 쓰이는 한국어 형태소 분석기
okt = Okt()                  # 형태소 분석기 객체 생성 (한 번만 만들면 계속 사용 가능)

print("✅ 준비 완료! 이제 CH 1 부터 시작하세요 🚀")

✅ 준비 완료! 이제 CH 1 부터 시작하세요 🚀


---
## 🤔 CH 1. 왜 전처리가 필요할까?

실제 텍스트 데이터는 생각보다 많이 **지저분**합니다.  
아래 리뷰들을 보면서 어떤 문제가 있는지 직접 확인해봅시다.

In [ ]:
# ── 리뷰 1번 ────────────────────────────────
# 특수문자와 이모티콘이 섞인 경우

리뷰1 = "ㅋㅋㅋ 진짜 맛있어요!!!! 근데 배송이 너무 느리네요ㅠㅠ"

# split() : 공백(띄어쓰기)을 기준으로 텍스트를 쪼개줍니다
단어들1 = 리뷰1.split()

print("원본 리뷰:", 리뷰1)
print("split() 결과:", 단어들1)
print("단어 수:", len(단어들1), "개")
print()
print("⚠️  문제점: 'ㅋㅋㅋ', '맛있어요!!!!', '느리네요ㅠㅠ' 처럼")
print("    기호가 단어에 붙어있어서 분석이 어렵습니다.")

원본 리뷰: ㅋㅋㅋ 진짜 맛있어요!!!! 근데 배송이 너무 느리네요ㅠㅠ
split() 결과: ['ㅋㅋㅋ', '진짜', '맛있어요!!!!', '근데', '배송이', '너무', '느리네요ㅠㅠ']
단어 수: 7 개

⚠️  문제점: 'ㅋㅋㅋ', '맛있어요!!!!', '느리네요ㅠㅠ' 처럼
    기호가 단어에 붙어있어서 분석이 어렵습니다.


In [ ]:
# ── 리뷰 2번 ────────────────────────────────
# HTML 태그와 URL이 섞인 경우

리뷰2 = "<b>최고</b>의 치킨!! http://naver.com 강추★★★ 가격도 저렴👍"

# split() : 공백을 기준으로 쪼개기
단어들2 = 리뷰2.split()

print("원본 리뷰:", 리뷰2)
print("split() 결과:", 단어들2)
print()
print("⚠️  문제점: '<b>최고</b>', 'http://naver.com', '강추★★★'")
print("    이런 것들이 단어로 인식됩니다. 분석에 전혀 도움이 안 돼요!")

원본 리뷰: <b>최고</b>의 치킨!! http://naver.com 강추★★★ 가격도 저렴👍
split() 결과: ['<b>최고</b>의', '치킨!!', 'http://naver.com', '강추★★★', '가격도', '저렴👍']

⚠️  문제점: '<b>최고</b>', 'http://naver.com', '강추★★★'
    이런 것들이 단어로 인식됩니다. 분석에 전혀 도움이 안 돼요!


In [ ]:
# ── 리뷰 3번 ────────────────────────────────
# 같은 의미인데 다양한 표현으로 쓰인 경우

리뷰3a = "맛잇다"   # 오탈자
리뷰3b = "맛있어요"  # 격식체
리뷰3c = "맛있다"   # 기본형

# 사람이 보기엔 모두 같은 말이지만
# 컴퓨터는 이 세 단어를 완전히 '다른 단어'로 인식합니다!

print("같은 의미인데...")
print("  단어 A:", 리뷰3a)
print("  단어 B:", 리뷰3b)
print("  단어 C:", 리뷰3c)
print()

# == 연산자 : 두 값이 같으면 True, 다르면 False
print("컴퓨터 입장에서 A == B:", 리뷰3a == 리뷰3b)  # False!
print("컴퓨터 입장에서 B == C:", 리뷰3b == 리뷰3c)  # False!
print()
print("⚠️  문제점: 같은 뜻이어도 표현이 다르면 컴퓨터는 다른 단어로 봅니다.")
print("    정규화(Normalization)로 이 문제를 해결합니다!")

같은 의미인데...
  단어 A: 맛잇다
  단어 B: 맛있어요
  단어 C: 맛있다

컴퓨터 입장에서 A == B: False
컴퓨터 입장에서 B == C: False

⚠️  문제점: 같은 뜻이어도 표현이 다르면 컴퓨터는 다른 단어로 봅니다.
    정규화(Normalization)로 이 문제를 해결합니다!


**💡 정리**  
전처리가 필요한 이유 3가지:
1. 특수문자/이모티콘이 분석을 방해합니다
2. URL, HTML 태그 같은 쓸모없는 정보가 섞여 있습니다
3. 같은 의미라도 표현이 달라 컴퓨터가 다르게 인식합니다

---
## 🗺️ CH 2. 전처리 4단계 흐름

전처리는 다음 4단계 순서로 진행합니다:

```
원본  →  1️⃣ 정제  →  2️⃣ 정규화  →  3️⃣ 토큰화  →  4️⃣ 불용어 제거  →  완료!
```

아래에서 같은 문장이 각 단계를 거쳐 어떻게 변하는지 **한 단계씩** 확인해봅시다.

In [ ]:
# ── 원본 텍스트 준비 ─────────────────────────

원본 = "ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr"

print("📌 원본 텍스트")
print(" ", 원본)

📌 원본 텍스트
  ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr


In [ ]:
# ── 1️⃣ 정제 단계 ────────────────────────────
# 목표: URL, 특수문자 등 쓸모없는 것들을 제거합니다

# re.sub(패턴, 바꿀문자, 원본) : 패턴과 일치하는 부분을 바꿀문자로 교체
# r'http\S+' : http로 시작하는 URL 패턴
# ''         : 빈 문자열로 바꾸기 = 삭제
step1 = re.sub(r'http\S+', '', 원본)

# r'[^가-힣a-zA-Z0-9 ]' : 한글·영문·숫자·공백이 아닌 것을 모두 선택
# '' : 삭제
step1 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', step1)

# r'\s+' : 공백이 2개 이상 연속된 부분
# ' '    : 공백 1개로 교체
# .strip() : 앞뒤 공백 제거
step1 = re.sub(r'\s+', ' ', step1).strip()

print("1️⃣  정제 후:", step1)
print("   → URL과 특수문자가 사라졌습니다!")

1️⃣  정제 후: 진짜 맛있어요 배달이 너무 느려요
   → URL과 특수문자가 사라졌습니다!


In [ ]:
# ── 2️⃣ 정규화 단계 ──────────────────────────
# 목표: 반복되는 표현을 통일합니다 (ㅋㅋㅋㅋ → ㅋㅋ)

# r'(.)\1{2,}' : 같은 문자가 3번 이상 반복되는 패턴
#   (.)   : 아무 문자 1개
#   \1    : 바로 앞에서 잡은 그 문자와 동일한 문자
#   {2,}  : 2번 이상 더 반복 → 합치면 3번 이상 반복
# r'\1\1' : 그 문자를 2번으로 줄임
step2 = re.sub(r'(.)\1{2,}', r'\1\1', step1)

print("2️⃣  정규화 후:", step2)
print("   → 'ㅋㅋㅋ' 같은 반복 문자가 압축되었습니다!")

2️⃣  정규화 후: 진짜 맛있어요 배달이 너무 느려요
   → 'ㅋㅋㅋ' 같은 반복 문자가 압축되었습니다!


In [ ]:
# ── 3️⃣ 토큰화 단계 ──────────────────────────
# 목표: 텍스트를 단어 단위로 쪼갭니다
# okt.nouns() : 한국어 문장에서 명사만 뽑아줍니다

step3 = okt.nouns(step2)

print("3️⃣  토큰화 후:", step3)
print("   → 명사 단어들만 리스트 형태로 분리되었습니다!")

3️⃣  토큰화 후: ['진짜', '배달']
   → 명사 단어들만 리스트 형태로 분리되었습니다!


In [ ]:
# ── 4️⃣ 불용어 제거 단계 ─────────────────────
# 목표: 분석에 도움이 안 되는 단어를 제거합니다

# 제거할 단어 목록 (필요에 따라 추가 가능)
불용어 = ["것", "수", "나", "저", "그", "이", "좀", "더"]

# 리스트 컴프리헨션 : 조건을 만족하는 단어만 남기는 방법
# w not in 불용어 : 불용어 목록에 없는 단어만
# len(w) >= 2    : 2글자 이상인 단어만
step4 = [w for w in step3 if w not in 불용어 and len(w) >= 2]

print("4️⃣  불용어 제거 후:", step4)
print()
print("✅ 원본:", 원본)
print("✅ 최종:", step4)
print("   → 핵심 단어만 깔끔하게 남았습니다!")

4️⃣  불용어 제거 후: ['진짜', '배달']

✅ 원본: ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr
✅ 최종: ['진짜', '배달']
   → 핵심 단어만 깔끔하게 남았습니다!


---
## ✂️ CH 3. 토큰화 (Tokenization)

**토큰화** 란 텍스트를 작은 단위(토큰)로 쪼개는 작업입니다.

```
"나는 학교에서 공부했어요"  →  ["나는", "학교에서", "공부했어요"]
```

방법에 따라 결과가 달라집니다. 하나씩 비교해봅시다!

In [ ]:
# ── 방법 1: split() — 가장 기본적인 방법 ────────
# split() : 공백(띄어쓰기)을 기준으로 문자열을 쪼개줍니다

문장 = "나는 오늘 학교에서 파이썬을 배웠어요"

# 공백 기준으로 쪼개기
토큰들 = 문장.split()

print("원본:", 문장)
print("결과:", 토큰들)
print("개수:", len(토큰들), "개")

원본: 나는 오늘 학교에서 파이썬을 배웠어요
결과: ['나는', '오늘', '학교에서', '파이썬을', '배웠어요']
개수: 5 개


In [ ]:
# ── split()의 한계 확인 ──────────────────────
# 특수문자가 붙어있으면 제대로 분리가 안 됩니다

문장2 = "오늘 날씨가 정말 좋네요!!! 산책가고 싶다ㅠㅠ"

# split()으로 쪼개면?
결과 = 문장2.split()

print("원본:", 문장2)
print("결과:", 결과)
print()
print("⚠️  문제: '좋네요!!!' 에서 '!!!' 가 붙어있고")
print("   '싶다ㅠㅠ' 에서 'ㅠㅠ' 가 붙어있습니다.")

원본: 오늘 날씨가 정말 좋네요!!! 산책가고 싶다ㅠㅠ
결과: ['오늘', '날씨가', '정말', '좋네요!!!', '산책가고', '싶다ㅠㅠ']

⚠️  문제: '좋네요!!!' 에서 '!!!' 가 붙어있고
   '싶다ㅠㅠ' 에서 'ㅠㅠ' 가 붙어있습니다.


In [ ]:
# ── 방법 2: NLTK word_tokenize() — 영어 전용 ────
# 문장부호(!, ., ?)를 단어에서 자동으로 분리해줍니다

from nltk.tokenize import word_tokenize

영어문장 = "I love text mining! It's really amazing."

# word_tokenize() : 영어 단어 토큰화
결과 = word_tokenize(영어문장)

print("원본:", 영어문장)
print("결과:", 결과)
print()
print("✅ '!' '.' 같은 문장부호가 단어에서 분리됩니다!")

원본: I love text mining! It's really amazing.
결과: ['I', 'love', 'text', 'mining', '!', 'It', "'s", 'really', 'amazing', '.']

✅ '!' '.' 같은 문장부호가 단어에서 분리됩니다!


In [ ]:
# ── 방법 3: NLTK sent_tokenize() — 문장 단위 분리 ─
# 단어가 아닌 '문장' 단위로 쪼개줍니다

from nltk.tokenize import sent_tokenize

긴텍스트 = "오늘 날씨가 좋네요. 산책을 나가고 싶어요. 하지만 할 일이 많습니다."

# sent_tokenize() : 문장 토큰화
문장들 = sent_tokenize(긴텍스트)

print("원본:", 긴텍스트)
print()
print("결과:")

# 결과를 하나씩 번호 붙여서 출력
print("  문장 1:", 문장들[0])
print("  문장 2:", 문장들[1])
print("  문장 3:", 문장들[2])
print()
print("총", len(문장들), "개의 문장으로 분리!")

원본: 오늘 날씨가 좋네요. 산책을 나가고 싶어요. 하지만 할 일이 많습니다.

결과:
  문장 1: 오늘 날씨가 좋네요.
  문장 2: 산책을 나가고 싶어요.
  문장 3: 하지만 할 일이 많습니다.

총 3 개의 문장으로 분리!


In [ ]:
# ── 한국어 토큰화의 특별한 문제 ────────────────
# 한국어는 조사(은/는/이/가/을/를)가 단어에 붙습니다

한국어문장 = "나는 학교에 가고 싶은데 비가 와서 못 갔어요"

# split()으로 쪼개면?
결과 = 한국어문장.split()

print("원본:", 한국어문장)
print("split() 결과:", 결과)
print()
print("⚠️  '나는'  = '나' + '는(조사)'")
print("   '학교에' = '학교' + '에(조사)'")
print("   '비가'   = '비' + '가(조사)'")
print()
print("→ 조사가 붙어있어서 '나는'과 '나를'이 다른 단어로 인식됩니다!")
print("→ 이 문제는 '형태소 분석'으로 해결합니다. (CH 7에서!)")

원본: 나는 학교에 가고 싶은데 비가 와서 못 갔어요
split() 결과: ['나는', '학교에', '가고', '싶은데', '비가', '와서', '못', '갔어요']

⚠️  '나는'  = '나' + '는(조사)'
   '학교에' = '학교' + '에(조사)'
   '비가'   = '비' + '가(조사)'

→ 조사가 붙어있어서 '나는'과 '나를'이 다른 단어로 인식됩니다!
→ 이 문제는 '형태소 분석'으로 해결합니다. (CH 7에서!)


In [5]:
# ✏️ 실습 : 내 문장을 직접 토큰화해보세요!

# 아래 문장을 원하는 내용으로 바꿔보세요 ↓
나의_문장 = "파이썬 텍스트 분석이 생각보다 재미있어요"

# 방법 1: split()
방법1 = 나의_문장.split()
print("방법 1 - split():", 방법1)

# 방법 2: okt.morphs() — 형태소 분리 (CH 0에서 만든 okt 사용)
방법2 = okt.morphs(나의_문장)
print("방법 2 - morphs():", 방법2)

# 방법 3: okt.nouns() — 명사만 추출
방법3 = okt.nouns(나의_문장)
print("방법 3 - nouns():", 방법3)

print()
print("세 방법 중 어떤 결과가 가장 분석에 유용해 보이나요?")

방법 1 - split(): ['파이썬', '텍스트', '분석이', '생각보다', '재미있어요']
방법 2 - morphs(): ['파이썬', '텍스트', '분석', '이', '생각', '보다', '재미있어요']
방법 3 - nouns(): ['파이썬', '텍스트', '분석', '생각']

세 방법 중 어떤 결과가 가장 분석에 유용해 보이나요?


---
## 🧹 CH 4. 정제 (Cleaning)

**정제** 란 텍스트에서 분석에 방해되는 노이즈를 제거하는 작업입니다.

주요 도구: **정규표현식 (`re` 라이브러리)**

| 제거 대상 | 예시 |
|-----------|------|
| 특수문자 | `!`, `★`, `ㅠ`, `👍` |
| URL | `http://naver.com` |
| HTML 태그 | `<b>`, `</p>` |
| 연속 공백 | `"오늘   날씨"` → `"오늘 날씨"` |

In [ ]:
# ── 특수문자 제거 : 한글만 남기기 ───────────────

텍스트1 = "ㅋㅋㅋ 진짜 맛있어요!!!! 최고★★★"

# re.sub(패턴, 바꿀것, 원본)
# r'[^가-힣 ]'  : 한글(가~힣)과 공백( ) 이외의 모든 문자
# ''            : 빈 문자열 = 삭제
결과1 = re.sub(r'[^가-힣 ]', '', 텍스트1)

print("원본:", 텍스트1)
print("결과:", 결과1)

원본: ㅋㅋㅋ 진짜 맛있어요!!!! 최고★★★
결과:  진짜 맛있어요 최고


In [ ]:
# ── 특수문자 제거 : 한글 + 영문 + 숫자 남기기 ────
# 영어와 숫자도 분석에 필요한 경우에 사용합니다

텍스트2 = "Python 3.10은 정말 좋아요!! 버전 업그레이드★"

# r'[^가-힣a-zA-Z0-9 ]' :
#   가-힣  : 한글
#   a-z    : 영어 소문자
#   A-Z    : 영어 대문자
#   0-9    : 숫자
#   ' '    : 공백
#   ^ (맨앞): 위 목록에 없는 것 = 나머지 전부
결과2 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트2)

print("원본:", 텍스트2)
print("결과:", 결과2)

원본: Python 3.10은 정말 좋아요!! 버전 업그레이드★
결과: Python 310은 정말 좋아요 버전 업그레이드


In [6]:
# ── URL 제거 ─────────────────────────────────

텍스트3 = "이 사이트 강추해요! https://www.naver.com 여기서 구매했어요"

# r'http\S+' :
#   http  : 'http' 로 시작하는
#   \S+   : 공백이 아닌 문자가 1개 이상 이어지는 것
#   → 합치면 : http로 시작해서 공백 전까지의 모든 문자열 = URL
결과3 = re.sub(r'http\S+', '', 텍스트3)

# 결과에 연속 공백이 생겼을 수 있으므로 정리
결과3 = re.sub(r'\s+', ' ', 결과3).strip()

print("원본:", 텍스트3)
print("결과:", 결과3)

원본: 이 사이트 강추해요! https://www.naver.com 여기서 구매했어요
결과: 이 사이트 강추해요! 여기서 구매했어요


In [ ]:
# ── HTML 태그 제거 ────────────────────────────
# 웹페이지에서 텍스트를 수집하면 HTML 태그가 섞여 있습니다

텍스트4 = "<h1>상품 후기</h1><p>이 상품은 <b>정말</b> 좋았어요!</p>"

# r'<[^>]+>' :
#   <       : '<' 로 시작
#   [^>]+   : '>' 가 아닌 문자가 1개 이상
#   >       : '>' 로 끝
#   → 합치면 : <...> 형태의 HTML 태그 전체
결과4 = re.sub(r'<[^>]+>', '', 텍스트4)

print("원본:", 텍스트4)
print("결과:", 결과4)

원본: <h1>상품 후기</h1><p>이 상품은 <b>정말</b> 좋았어요!</p>
결과: 상품 후기이 상품은 정말 좋았어요!


In [ ]:
# ── 연속 공백 정리 ────────────────────────────
# 정제 과정에서 공백이 여러 개 생길 수 있습니다

텍스트5 = "안녕하세요   반갑습니다    오늘   날씨가    좋네요"

# r'\s+' : 공백 문자가 1개 이상 연속된 것
# ' '    : 공백 1개로 교체
# .strip() : 문자열 앞뒤의 공백 제거
결과5 = re.sub(r'\s+', ' ', 텍스트5).strip()

# repr() : 공백이 눈에 보이도록 표시해주는 함수
print("원본:", repr(텍스트5))
print("결과:", repr(결과5))

원본: '안녕하세요   반갑습니다    오늘   날씨가    좋네요'
결과: '안녕하세요 반갑습니다 오늘 날씨가 좋네요'


In [ ]:
# ── 지금까지 배운 정제 방법을 함수 하나로 묶기 ──
# def 함수이름(매개변수): 자주 쓰는 코드를 묶어서 재사용합니다

def 텍스트_정제(텍스트):
    # 1. URL 제거 (http로 시작하는 주소)
    텍스트 = re.sub(r'http\S+', '', 텍스트)
    # 2. HTML 태그 제거 (<...> 형태)
    텍스트 = re.sub(r'<[^>]+>', '', 텍스트)
    # 3. 특수문자 제거 (한글·영문·숫자·공백만 남김)
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)
    # 4. 연속 공백을 공백 1개로 정리
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()
    # 처리된 텍스트를 반환
    return 텍스트

print("✅ 정제 함수 준비 완료!")
print("   이제 아래 셀에서 실제로 사용해봅시다.")

✅ 정제 함수 준비 완료!
   이제 아래 셀에서 실제로 사용해봅시다.


In [ ]:
# ── 정제 함수 테스트 1 ───────────────────────

리뷰A = "ㅋㅋ 진짜 맛있어요!!!! http://naver.com 강추★"

# 만들어둔 함수에 리뷰를 넣기
정제A = 텍스트_정제(리뷰A)

print("원본:", 리뷰A)
print("결과:", 정제A)

원본: ㅋㅋ 진짜 맛있어요!!!! http://naver.com 강추★
결과: 진짜 맛있어요 강추


In [ ]:
# ── 정제 함수 테스트 2 ───────────────────────

리뷰B = "<b>배송</b>이 너무 느려요 😤😤 별로임"

정제B = 텍스트_정제(리뷰B)

print("원본:", 리뷰B)
print("결과:", 정제B)

원본: <b>배송</b>이 너무 느려요 😤😤 별로임
결과: 배송이 너무 느려요 별로임


In [ ]:
# ── 정제 함수 테스트 3 ───────────────────────

리뷰C = "가격이   너무   비싸요...   다시는 안살듯"

정제C = 텍스트_정제(리뷰C)

print("원본:", 리뷰C)
print("결과:", 정제C)

원본: 가격이   너무   비싸요...   다시는 안살듯
결과: 가격이 너무 비싸요 다시는 안살듯


In [ ]:
# ✏️ 실습 : 직접 만든 리뷰를 정제해보세요!

# 아래 텍스트를 원하는 내용으로 바꿔보세요 ↓
나의_리뷰 = "와!!!!!! <div>이 제품</div> 진짜 대박 https://coupang.com 👍👍👍"

# 정제 함수 적용
나의_결과 = 텍스트_정제(나의_리뷰)

print("원본:", 나의_리뷰)
print("결과:", 나의_결과)
print()
# 체크리스트 자동 확인
print("체크리스트")
print("  URL 제거됨:", "http" not in 나의_결과)  # True면 제거 성공
print("  HTML 제거됨:", "<" not in 나의_결과)     # True면 제거 성공
print("  특수문자 제거됨:", "!" not in 나의_결과) # True면 제거 성공

원본: 와!!!!!! <div>이 제품</div> 진짜 대박 https://coupang.com 👍👍👍
결과: 와 이 제품 진짜 대박

체크리스트
  URL 제거됨: True
  HTML 제거됨: True
  특수문자 제거됨: True


---
## 🔧 CH 5. 정규화 (Normalization)

**정규화** 란 같은 의미를 가진 다양한 표현을 **하나의 형태로 통일**하는 작업입니다.

```
"Python" = "PYTHON" = "python"  →  모두 "python" 으로 통일
"ㅋㅋㅋㅋㅋ" = "ㅋㅋㅋ" = "ㅋㅋ"   →  모두 "ㅋㅋ" 로 통일
"먹었어요" = "먹었다" = "먹습니다" →  모두 "먹다" 로 통일
```

In [ ]:
# ── 영어 대소문자 → 소문자 통일 ─────────────────
# .lower() : 문자열의 모든 영문자를 소문자로 바꿉니다

단어A = "Python"
단어B = "PYTHON"
단어C = "PyThOn"

print("소문자 변환 전:", 단어A, "|", 단어B, "|", 단어C)
print("소문자 변환 후:", 단어A.lower(), "|", 단어B.lower(), "|", 단어C.lower())

소문자 변환 전: Python | PYTHON | PyThOn
소문자 변환 후: python | python | python


In [ ]:
# ── 대소문자 통일이 왜 중요한가? ─────────────────
# 같은 단어를 다르게 인식하는 문제를 직접 확인!

텍스트 = "I love Python and PYTHON is great because python is easy"
단어_목록 = 텍스트.split()  # 공백으로 쪼개기

print("정규화 전 - 각각 몇 번 나왔나?")
# .count() : 리스트에서 특정 값이 몇 번 나오는지 셉니다
print("  'Python' 횟수:", 단어_목록.count('Python'))
print("  'PYTHON' 횟수:", 단어_목록.count('PYTHON'))
print("  'python' 횟수:", 단어_목록.count('python'))
print()

# 소문자로 통일 후
# 리스트 컴프리헨션 : [표현식 for 변수 in 리스트]
소문자_목록 = [단어.lower() for 단어 in 단어_목록]

print("정규화 후 - 소문자 'python' 몇 번?")
print("  'python' 전체 횟수:", 소문자_목록.count('python'))
print("  → 세 단어가 하나로 합쳐졌습니다!")

정규화 전 - 각각 몇 번 나왔나?
  'Python' 횟수: 1
  'PYTHON' 횟수: 1
  'python' 횟수: 1

정규화 후 - 소문자 'python' 몇 번?
  'python' 전체 횟수: 3
  → 세 단어가 하나로 합쳐졌습니다!


In [ ]:
# ── 반복 문자 압축 함수 만들기 ───────────────────

def 반복문자_정규화(텍스트):
    # r'(.)\1{2,}' : 같은 문자가 3번 이상 반복되는 패턴을 찾기
    # r'\1\1'      : 찾은 문자를 2번으로 줄이기
    return re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

print("✅ 반복문자 정규화 함수 준비 완료!")

✅ 반복문자 정규화 함수 준비 완료!


In [ ]:
# ── 반복 문자 압축 : ㅋㅋㅋㅋ → ㅋㅋ ───────────

텍스트A = "ㅋㅋㅋㅋㅋㅋ 너무 웃겨요"
결과A = 반복문자_정규화(텍스트A)

print("원본:", 텍스트A)
print("결과:", 결과A)

원본: ㅋㅋㅋㅋㅋㅋ 너무 웃겨요
결과: ㅋㅋ 너무 웃겨요


In [ ]:
# ── 반복 문자 압축 : ㅠㅠㅠㅠ → ㅠㅠ ───────────

텍스트B = "맛있어요ㅠㅠㅠㅠㅠ 최고야"
결과B = 반복문자_정규화(텍스트B)

print("원본:", 텍스트B)
print("결과:", 결과B)

원본: 맛있어요ㅠㅠㅠㅠㅠ 최고야
결과: 맛있어요ㅠㅠ 최고야


In [ ]:
# ── 반복 문자 압축 : !!!!! → !! ─────────────────

텍스트C = "와!!!!!!!! 대박이다"
결과C = 반복문자_정규화(텍스트C)

print("원본:", 텍스트C)
print("결과:", 결과C)

원본: 와!!!!!!!! 대박이다
결과: 와!! 대박이다


In [ ]:
# ── 어간 추출 : 동사/형용사의 원형 복원 ──────────
# stem=True : 활용형을 원형으로 바꿉니다
# 예) 먹었어요 → 먹다 / 재미있었고 → 재미있다

문장 = "그 영화는 정말 재미있었고 배우 연기도 훌륭했어요"

# stem 옵션 없이 (기본)
기본 = okt.morphs(문장)
print("stem=False (기본):", 기본)

# stem=True 적용
어간 = okt.morphs(문장, stem=True)
print("stem=True (어간) :", 어간)
print()
print("→ '재미있었고' 가 '재미있다'로, '훌륭했어요'가 '훌륭하다'로 바뀝니다!")

stem=False (기본): ['그', '영화', '는', '정말', '재미있었고', '배우', '연기', '도', '훌륭했어요']
stem=True (어간) : ['그', '영화', '는', '정말', '재미있다', '배우', '연기', '도', '훌륭하다']

→ '재미있었고' 가 '재미있다'로, '훌륭했어요'가 '훌륭하다'로 바뀝니다!


In [ ]:
# ✏️ 실습 : 정규화 두 가지를 직접 적용해보세요!

나의_텍스트 = "PYTHON 진짜 재미있어요ㅋㅋㅋㅋㅋ 완전 대박!!!!!!!"

print("원본:", 나의_텍스트)

# 1단계: 영어를 소문자로 변환
단계1 = 나의_텍스트.lower()
print("소문자 변환:", 단계1)

# 2단계: 반복 문자 압축
단계2 = 반복문자_정규화(단계1)
print("반복 압축:", 단계2)

원본: PYTHON 진짜 재미있어요ㅋㅋㅋㅋㅋ 완전 대박!!!!!!!
소문자 변환: python 진짜 재미있어요ㅋㅋㅋㅋㅋ 완전 대박!!!!!!!
반복 압축: python 진짜 재미있어요ㅋㅋ 완전 대박!!


---
## 🗑️ CH 6. 불용어 처리 (Stopwords)

**불용어** 란 분석에 별 도움이 안 되는 단어들입니다.

```
"저는  오늘  정말  맛있는  피자를  먹었습니다"
  ↑           ↑                    ↑
 불용어       불용어               불용어  ← 제거!

→ 남는 단어: "오늘"  "맛있는"  "피자"  "먹다"
```

In [ ]:
# ── 영어 불용어 목록 확인 (NLTK 제공) ─────────────
# NLTK에는 이미 179개의 영어 불용어가 준비되어 있습니다

from nltk.corpus import stopwords

# 영어 불용어 목록 불러오기
영어_불용어 = stopwords.words('english')

print("영어 불용어 개수:", len(영어_불용어), "개")
print()
# 처음 20개만 보기 (슬라이싱: 리스트[시작:끝])
print("예시 (앞 20개):", 영어_불용어[:20])

영어 불용어 개수: 198 개

예시 (앞 20개): ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


In [ ]:
# ── 영어 문장에 불용어 제거 적용 ─────────────────

from nltk.tokenize import word_tokenize

영어문장 = "This movie is really good and I enjoyed watching it very much"

# 1) 소문자로 통일 후 토큰화
토큰들 = word_tokenize(영어문장.lower())
print("토큰화 결과:", 토큰들)
print("단어 수:", len(토큰들), "개")
print()

# 2) 불용어 제거
# 리스트 컴프리헨션 : 불용어 목록에 없는 단어만 남기기
핵심단어들 = [토큰 for 토큰 in 토큰들 if 토큰 not in 영어_불용어]
print("불용어 제거 후:", 핵심단어들)
print("단어 수:", len(핵심단어들), "개")
print()
print("→", len(토큰들), "개에서", len(핵심단어들), "개로! 핵심 단어만 남았습니다.")

토큰화 결과: ['this', 'movie', 'is', 'really', 'good', 'and', 'i', 'enjoyed', 'watching', 'it', 'very', 'much']
단어 수: 12 개

불용어 제거 후: ['movie', 'really', 'good', 'enjoyed', 'watching', 'much']
단어 수: 6 개

→ 12 개에서 6 개로! 핵심 단어만 남았습니다.


In [ ]:
# ── 한국어 불용어 목록 만들기 ────────────────────
# 한국어는 NLTK에서 제공하지 않아서 직접 만들어야 합니다!

한국어_불용어 = [
    # 조사 (단어에 붙어서 문법적 의미를 추가하는 말)
    "이", "가", "은", "는", "을", "를", "의",
    "에", "에서", "로", "으로", "과", "와", "도", "만",
    # 접속사 (문장을 연결하는 말)
    "그리고", "그러나", "그런데", "그래서", "하지만",
    # 정도 부사 (의미는 있지만 분석 가치가 낮은 말)
    "정말", "진짜", "매우", "너무", "아주", "좀",
    # 기타 불필요 단어
    "것", "수", "있다", "없다", "합니다", "입니다"
]

print("한국어 불용어 개수:", len(한국어_불용어), "개")
print("목록:", 한국어_불용어)

한국어 불용어 개수: 32 개
목록: ['이', '가', '은', '는', '을', '를', '의', '에', '에서', '로', '으로', '과', '와', '도', '만', '그리고', '그러나', '그런데', '그래서', '하지만', '정말', '진짜', '매우', '너무', '아주', '좀', '것', '수', '있다', '없다', '합니다', '입니다']


In [ ]:
# ── 한국어 문장에 불용어 제거 적용 ───────────────

문장 = "이 음식은 정말 맛있고 가격도 매우 저렴합니다"

# 1) split() 으로 토큰화
토큰들 = 문장.split()
print("토큰화:", 토큰들)
print()

# 2) 불용어 제거
결과 = [토큰 for 토큰 in 토큰들 if 토큰 not in 한국어_불용어]
print("불용어 제거 후:", 결과)
print()
print("⚠️  아직 '음식은', '가격도' 처럼 조사가 붙어 있어요.")
print("   형태소 분석(CH 7)을 먼저 하면 더 깔끔해집니다!")

토큰화: ['이', '음식은', '정말', '맛있고', '가격도', '매우', '저렴합니다']

불용어 제거 후: ['음식은', '맛있고', '가격도', '저렴합니다']

⚠️  아직 '음식은', '가격도' 처럼 조사가 붙어 있어요.
   형태소 분석(CH 7)을 먼저 하면 더 깔끔해집니다!


In [ ]:
# ── 리스트 컴프리헨션 이해하기 ───────────────────
# [표현식 for 변수 in 리스트 if 조건]
# → 리스트의 각 항목을 꺼내서, 조건을 만족하면 결과에 포함

단어들 = ["나는", "정말", "맛있는", "밥을", "먹었다"]
불용어 = ["나는", "정말"]

# 방법 1: 일반 for문 (이해하기 쉬운 방법)
결과1 = []                          # 빈 리스트 준비
for 단어 in 단어들:                  # 단어를 하나씩 꺼내서
    if 단어 not in 불용어:           # 불용어 목록에 없으면
        결과1.append(단어)           # 결과 리스트에 추가

# 방법 2: 리스트 컴프리헨션 (짧고 빠른 방법)
결과2 = [단어 for 단어 in 단어들 if 단어 not in 불용어]

print("일반 for문:", 결과1)
print("리스트 컴프리헨션:", 결과2)
print("결과가 같나요?", 결과1 == 결과2)
print()
print("두 방법은 완전히 같습니다! 처음엔 방법 1로 이해하고, 익숙해지면 방법 2를 쓰세요.")

일반 for문: ['맛있는', '밥을', '먹었다']
리스트 컴프리헨션: ['맛있는', '밥을', '먹었다']
결과가 같나요? True

두 방법은 완전히 같습니다! 처음엔 방법 1로 이해하고, 익숙해지면 방법 2를 쓰세요.


In [ ]:
# ✏️ 실습 : 나만의 불용어 목록 만들고 적용하기!

# 아래 목록에 원하는 단어를 추가해보세요 ↓
나의_불용어 = ["은", "는", "이", "가", "을", "를"]
# 예: "정말", "진짜", "너무" 를 추가하면 어떻게 달라질까요?

리뷰 = "이 식당은 정말 맛있는 음식이 많고 서비스도 정말 좋아요"
토큰들 = 리뷰.split()

결과 = [토큰 for 토큰 in 토큰들 if 토큰 not in 나의_불용어]

print("원본:", 리뷰)
print("토큰:", 토큰들)
print("결과:", 결과)
print()
print("불용어를 더 추가하면 결과가 어떻게 달라지는지 실험해보세요!")

원본: 이 식당은 정말 맛있는 음식이 많고 서비스도 정말 좋아요
토큰: ['이', '식당은', '정말', '맛있는', '음식이', '많고', '서비스도', '정말', '좋아요']
결과: ['식당은', '정말', '맛있는', '음식이', '많고', '서비스도', '정말', '좋아요']

불용어를 더 추가하면 결과가 어떻게 달라지는지 실험해보세요!


---
## 🇰🇷 CH 7. 한국어 형태소 분석 — KoNLPy

**형태소** 란 의미를 가진 가장 작은 언어 단위입니다.

```
"학교에서" → "학교" + "에서"    (명사 + 조사)
"맛있었어요" → "맛있" + "었" + "어요"  (형용사 + 과거 + 어미)
```

**KoNLPy** 는 한국어 형태소 분석을 해주는 Python 라이브러리입니다.  
주로 사용하는 메서드 4가지를 차례로 배웁니다.

In [ ]:
# ── morphs() : 형태소 단위로 쪼개기 ────────────
# 단어를 조사, 어미까지 분리해서 리스트로 반환합니다

문장 = "나는 오늘 학교에서 파이썬을 배웠어요"

# okt.morphs() : 형태소 분리
형태소들 = okt.morphs(문장)

print("원본:", 문장)
print("형태소:", 형태소들)
print()
print("→ '나는' = '나' + '는', '학교에서' = '학교' + '에서' 로 분리됩니다!")

원본: 나는 오늘 학교에서 파이썬을 배웠어요
형태소: ['나', '는', '오늘', '학교', '에서', '파이썬', '을', '배웠어요']

→ '나는' = '나' + '는', '학교에서' = '학교' + '에서' 로 분리됩니다!


In [ ]:
# ── pos() : 품사 태깅 (형태소 + 품사) ───────────
# 각 형태소가 어떤 품사인지 함께 알려줍니다

문장 = "나는 오늘 학교에서 파이썬을 배웠어요"

# okt.pos() : 각 형태소와 품사를 (단어, 품사) 쌍으로 반환
품사_결과 = okt.pos(문장)

print("원본:", 문장)
print("품사 태깅 결과:")
print(품사_결과)
print()

# 품사 코드 설명
print("품사 코드 설명:")
print("  Noun       = 명사")
print("  Josa       = 조사")
print("  Verb       = 동사")
print("  Adjective  = 형용사")
print("  Adverb     = 부사")

원본: 나는 오늘 학교에서 파이썬을 배웠어요
품사 태깅 결과:
[('나', 'Noun'), ('는', 'Josa'), ('오늘', 'Noun'), ('학교', 'Noun'), ('에서', 'Josa'), ('파이썬', 'Noun'), ('을', 'Josa'), ('배웠어요', 'Verb')]

품사 코드 설명:
  Noun       = 명사
  Josa       = 조사
  Verb       = 동사
  Adjective  = 형용사
  Adverb     = 부사


In [ ]:
# ── pos() 결과를 표 형태로 예쁘게 출력하기 ────────

문장2 = "예쁜 카페에서 맛있는 커피를 마셨어요"

품사_설명 = {
    'Noun': '명사', 'Verb': '동사', 'Adjective': '형용사',
    'Adverb': '부사', 'Josa': '조사', 'Eomi': '어미',
    'Alpha': '영문자', 'Punctuation': '문장부호'
}

print(f"문장: {문장2}")
print()
# f-string의 {:8} : 8칸 고정폭으로 출력 (표처럼 정렬)
print(f"{'단어':10} {'품사코드':15} {'품사설명'}")
print("-" * 40)

for 단어, 품사 in okt.pos(문장2):
    # .get(키, 기본값) : 딕셔너리에서 키를 찾고, 없으면 기본값 반환
    설명 = 품사_설명.get(품사, 품사)
    print(f"  {단어:10} {품사:15} {설명}")

문장: 예쁜 카페에서 맛있는 커피를 마셨어요

단어         품사코드            품사설명
----------------------------------------
  예쁜         Adjective       형용사
  카페         Noun            명사
  에서         Josa            조사
  맛있는        Adjective       형용사
  커피         Noun            명사
  를          Josa            조사
  마셨어요       Verb            동사


In [ ]:
# ── nouns() : 명사만 추출 ── ★ 가장 많이 쓰입니다! ★
# 텍스트의 핵심 의미는 명사에 담겨 있는 경우가 많습니다

문장3 = "오늘 배달 시킨 치킨이 정말 바삭하고 맛있었어요. 소스도 최고!"

# okt.nouns() : 명사만 추출해서 리스트로 반환
명사들 = okt.nouns(문장3)

print("원본:", 문장3)
print("명사:", 명사들)
print()
print("→ 핵심 단어만 딱 남습니다!")

원본: 오늘 배달 시킨 치킨이 정말 바삭하고 맛있었어요. 소스도 최고!
명사: ['오늘', '배달', '치킨', '정말', '바삭', '소스', '최고']

→ 핵심 단어만 딱 남습니다!


In [ ]:
# ── morphs(stem=True) : 어간 추출 ─────────────
# 다양한 활용형을 원형으로 통일합니다

# 같은 의미, 다른 표현 3가지
문장_A = "그 영화는 재미있었어요"   # 과거형
문장_B = "그 영화는 재미있어요"    # 현재형
문장_C = "그 영화는 재미있습니다"  # 격식체

print("[stem=False (기본)]")
print("  A:", okt.morphs(문장_A))
print("  B:", okt.morphs(문장_B))
print("  C:", okt.morphs(문장_C))
print()

print("[stem=True (원형으로 통일)]")
print("  A:", okt.morphs(문장_A, stem=True))
print("  B:", okt.morphs(문장_B, stem=True))
print("  C:", okt.morphs(문장_C, stem=True))
print()
print("→ 세 문장 모두 '재미있다'로 통일됩니다!")

[stem=False (기본)]
  A: ['그', '영화', '는', '재미있었어요']
  B: ['그', '영화', '는', '재미있어요']
  C: ['그', '영화', '는', '재미있습니다']

[stem=True (원형으로 통일)]
  A: ['그', '영화', '는', '재미있다']
  B: ['그', '영화', '는', '재미있다']
  C: ['그', '영화', '는', '재미있다']

→ 세 문장 모두 '재미있다'로 통일됩니다!


In [ ]:
# ✏️ 실습 : 리뷰에서 핵심 명사 뽑기!

나의_리뷰 = "오늘 저녁에 시킨 피자가 정말 맛있었어요! 도우가 쫄깃하고 치즈가 풍부해서 최고였습니다"

print("리뷰:", 나의_리뷰)
print()

# 1) 전체 형태소
전체 = okt.morphs(나의_리뷰)
print("1) 전체 형태소:", 전체)

# 2) 명사만
명사 = okt.nouns(나의_리뷰)
print("2) 명사만:", 명사)

# 3) 2글자 이상 명사만 (너무 짧은 단어는 의미가 적습니다)
# len(단어) >= 2 : 글자 수가 2개 이상인 것만
긴_명사 = [단어 for 단어 in 명사 if len(단어) >= 2]
print("3) 2글자 이상 명사:", 긴_명사)

# 4) 단어 빈도 계산
# Counter() : 리스트에서 각 항목이 몇 번 나오는지 세어줍니다
빈도 = Counter(긴_명사)
print()
print("자주 등장한 단어 TOP 3:", 빈도.most_common(3))

리뷰: 오늘 저녁에 시킨 피자가 정말 맛있었어요! 도우가 쫄깃하고 치즈가 풍부해서 최고였습니다

1) 전체 형태소: ['오늘', '저녁', '에', '시킨', '피자', '가', '정말', '맛있었어요', '!', '도우', '가', '쫄깃', '하고', '치즈', '가', '풍부해서', '최고', '였습니다']
2) 명사만: ['오늘', '저녁', '피자', '정말', '도우', '쫄깃', '치즈', '최고']
3) 2글자 이상 명사: ['오늘', '저녁', '피자', '정말', '도우', '쫄깃', '치즈', '최고']

자주 등장한 단어 TOP 3: [('오늘', 1), ('저녁', 1), ('피자', 1)]


---
## 🔗 CH 8. 파이프라인 통합 실습

지금까지 배운 모든 단계를 **하나의 함수**로 합쳐봅니다!

```
원본 → 정제 → 정규화 → 형태소분석 → 불용어제거 → 완료!
```

In [ ]:
# ── 전처리 파이프라인 함수 정의 ─────────────────
# 모든 단계를 하나로 묶은 함수입니다

# 분석에 불필요한 한국어 단어 목록
불용어_목록 = [
    "것", "수", "나", "저", "제", "그", "이", "때",
    "등", "좀", "잘", "더", "한", "안", "못", "또"
]

def 전처리_파이프라인(텍스트):
    """
    입력: 원본 텍스트 (문자열)
    출력: 전처리된 핵심 단어 리스트
    """
    # 1단계: 정제 — 노이즈 제거
    텍스트 = re.sub(r'http\S+', '', 텍스트)               # URL 제거
    텍스트 = re.sub(r'<[^>]+>', '', 텍스트)               # HTML 태그 제거
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)    # 특수문자 제거
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()           # 공백 정리

    # 2단계: 정규화 — 반복 문자 압축
    텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

    # 3단계: 형태소 분석 — 명사 추출
    토큰들 = okt.nouns(텍스트)

    # 4단계: 불용어 제거 + 2글자 이상만 유지
    결과 = [
        단어 for 단어 in 토큰들
        if 단어 not in 불용어_목록  # 불용어 목록에 없는 것만
        and len(단어) >= 2          # 2글자 이상인 것만
    ]

    return 결과

print("✅ 파이프라인 함수 준비 완료!")

✅ 파이프라인 함수 준비 완료!


In [ ]:
# ── 파이프라인 테스트 1 ──────────────────────

리뷰1 = "ㅋㅋㅋ 진짜 맛있어요!!!! http://naver.com 치킨 배달 강추★★★"
결과1 = 전처리_파이프라인(리뷰1)

print("원본:", 리뷰1)
print("결과:", 결과1)

원본: ㅋㅋㅋ 진짜 맛있어요!!!! http://naver.com 치킨 배달 강추★★★
결과: ['진짜', '치킨', '배달', '강추']


In [ ]:
# ── 파이프라인 테스트 2 ──────────────────────

리뷰2 = "배송이 너무 느려서 음식이 다 식었어요ㅠㅠㅠ 다시는 안 시킬듯"
결과2 = 전처리_파이프라인(리뷰2)

print("원본:", 리뷰2)
print("결과:", 결과2)

원본: 배송이 너무 느려서 음식이 다 식었어요ㅠㅠㅠ 다시는 안 시킬듯
결과: ['배송', '음식']


In [ ]:
# ── 파이프라인 테스트 3 ──────────────────────

리뷰3 = "가격 대비 양도 많고 맛도 괜찮았습니다. 재주문 의사 100% 있습니다."
결과3 = 전처리_파이프라인(리뷰3)

print("원본:", 리뷰3)
print("결과:", 결과3)

원본: 가격 대비 양도 많고 맛도 괜찮았습니다. 재주문 의사 100% 있습니다.
결과: ['가격', '대비', '양도', '주문', '의사']


In [ ]:
# ── 파이프라인 테스트 4 ──────────────────────

리뷰4 = "포장이 예쁘고 서비스도 훌륭해요!! 친절한 사장님 덕분에 기분 좋았어요"
결과4 = 전처리_파이프라인(리뷰4)

print("원본:", 리뷰4)
print("결과:", 결과4)

원본: 포장이 예쁘고 서비스도 훌륭해요!! 친절한 사장님 덕분에 기분 좋았어요
결과: ['포장', '서비스', '사장', '덕분', '기분']


In [ ]:
# ── 파이프라인 테스트 5 ──────────────────────

리뷰5 = "솔직히 기대보다 별로였어요. 홍보랑 실제 음식이 많이 달랐네요."
결과5 = 전처리_파이프라인(리뷰5)

print("원본:", 리뷰5)
print("결과:", 결과5)

원본: 솔직히 기대보다 별로였어요. 홍보랑 실제 음식이 많이 달랐네요.
결과: ['기대', '별로', '홍보', '실제', '음식']


In [ ]:
# ── 전처리 결과로 단어 빈도 분석 ────────────────
# 5개 리뷰의 단어를 합쳐서 가장 많이 나온 단어를 찾습니다

# extend() : 리스트에 다른 리스트의 항목을 모두 추가
모든_단어 = []
모든_단어.extend(결과1)  # 리뷰1 단어 추가
모든_단어.extend(결과2)  # 리뷰2 단어 추가
모든_단어.extend(결과3)  # 리뷰3 단어 추가
모든_단어.extend(결과4)  # 리뷰4 단어 추가
모든_단어.extend(결과5)  # 리뷰5 단어 추가

print("전체 단어 목록:", 모든_단어)
print("전체 단어 수:", len(모든_단어), "개")

전체 단어 목록: ['진짜', '치킨', '배달', '강추', '배송', '음식', '가격', '대비', '양도', '주문', '의사', '포장', '서비스', '사장', '덕분', '기분', '기대', '별로', '홍보', '실제', '음식']
전체 단어 수: 21 개


In [ ]:
# ── TOP 10 단어 출력 (막대 그래프 포함) ──────────

# Counter() : 각 단어가 몇 번 나왔는지 세어줍니다
빈도 = Counter(모든_단어)

print("단어 빈도 TOP 10")
print()
print(f"{'순위':4} {'단어':10} {'횟수':5}  막대")
print("-" * 35)

# .most_common(10) : 가장 많이 나온 순서로 10개 반환
for 순위, (단어, 횟수) in enumerate(빈도.most_common(10), 1):
    막대 = "█" * 횟수  # 횟수만큼 블록 기호 반복
    print(f"  {순위:2}위  {단어:10} {횟수:3}회  {막대}")

print()
print("💡 다음 주에는 이 결과를 TF-IDF로 더 정교하게 분석합니다!")

단어 빈도 TOP 10

순위   단어         횟수     막대
-----------------------------------
   1위  음식           2회  ██
   2위  진짜           1회  █
   3위  치킨           1회  █
   4위  배달           1회  █
   5위  강추           1회  █
   6위  배송           1회  █
   7위  가격           1회  █
   8위  대비           1회  █
   9위  양도           1회  █
  10위  주문           1회  █

💡 다음 주에는 이 결과를 TF-IDF로 더 정교하게 분석합니다!


---
## 🏆 CH 9. 도전 과제

지금까지 배운 내용을 활용해 **영화 리뷰 분석**을 완성해보세요!  
막히면 위의 CH 8 코드를 참고하세요.

**목표:**
1. 영화 리뷰 5개를 파이프라인으로 전처리
2. 단어를 모두 합쳐서 빈도 분석
3. TOP 5 단어 출력

In [ ]:
# ── 과제용 영화 리뷰 데이터 ──────────────────

영화리뷰1 = "ㅋㅋㅋ 이 영화 진짜 대박이에요!!! 배우들 연기가 최고였고 스토리도 탄탄했어요 👍"
영화리뷰2 = "스포일러 주의... 결말이 너무 허무했어요ㅠㅠ 2시간이 아깝다는 생각이 들었습니다"
영화리뷰3 = "CGV에서 봤는데 영상미가 정말 뛰어났어요 https://movie.naver.com/12345 강추합니다"
영화리뷰4 = "배우 연기는 좋은데 각본이 너무 진부해서 아쉬웠어요. 그래도 볼 만은 해요."
영화리뷰5 = "역대급 영화!! 올해 본 영화 중에 단연 최고!! 꼭 보세요 꼭꼭꼭!!!!!"

print("영화 리뷰 5개 준비 완료!")
print()
print("1:", 영화리뷰1)
print("2:", 영화리뷰2)
print("3:", 영화리뷰3)
print("4:", 영화리뷰4)
print("5:", 영화리뷰5)

영화 리뷰 5개 준비 완료!

1: ㅋㅋㅋ 이 영화 진짜 대박이에요!!! 배우들 연기가 최고였고 스토리도 탄탄했어요 👍
2: 스포일러 주의... 결말이 너무 허무했어요ㅠㅠ 2시간이 아깝다는 생각이 들었습니다
3: CGV에서 봤는데 영상미가 정말 뛰어났어요 https://movie.naver.com/12345 강추합니다
4: 배우 연기는 좋은데 각본이 너무 진부해서 아쉬웠어요. 그래도 볼 만은 해요.
5: 역대급 영화!! 올해 본 영화 중에 단연 최고!! 꼭 보세요 꼭꼭꼭!!!!!


In [ ]:
# ✏️ STEP 1: 각 리뷰를 파이프라인으로 전처리하세요
# 힌트: 전처리_파이프라인(리뷰) 형태로 사용합니다

# 리뷰1 전처리
전처리1 = 전처리_파이프라인(영화리뷰1)
print("리뷰1 전처리:", 전처리1)

# 리뷰2 전처리
전처리2 = 전처리_파이프라인(영화리뷰2)
print("리뷰2 전처리:", 전처리2)

# 리뷰3 전처리
전처리3 = 전처리_파이프라인(영화리뷰3)
print("리뷰3 전처리:", 전처리3)

# 리뷰4 전처리
전처리4 = 전처리_파이프라인(영화리뷰4)
print("리뷰4 전처리:", 전처리4)

# 리뷰5 전처리
전처리5 = 전처리_파이프라인(영화리뷰5)
print("리뷰5 전처리:", 전처리5)

리뷰1 전처리: ['영화', '진짜', '대박', '배우', '연기', '최고', '스토리']
리뷰2 전처리: ['스포일러', '주의', '결말', '생각']
리뷰3 전처리: ['영상', '미가', '정말']
리뷰4 전처리: ['배우', '연기', '각본']
리뷰5 전처리: ['역대', '영화', '올해', '영화', '단연', '최고', '꼭꼭']


In [ ]:
# ✏️ STEP 2: 5개 리뷰의 단어를 하나의 리스트로 합치세요
# 힌트: extend() 를 사용합니다

영화_단어들 = []  # 빈 리스트 준비

영화_단어들.extend(전처리1)  # 리뷰1 단어 추가
영화_단어들.extend(전처리2)  # 리뷰2 단어 추가
영화_단어들.extend(전처리3)  # 리뷰3 단어 추가
영화_단어들.extend(전처리4)  # 리뷰4 단어 추가
영화_단어들.extend(전처리5)  # 리뷰5 단어 추가

print("전체 단어 목록:", 영화_단어들)
print("총 단어 수:", len(영화_단어들), "개")

전체 단어 목록: ['영화', '진짜', '대박', '배우', '연기', '최고', '스토리', '스포일러', '주의', '결말', '생각', '영상', '미가', '정말', '배우', '연기', '각본', '역대', '영화', '올해', '영화', '단연', '최고', '꼭꼭']
총 단어 수: 24 개


In [ ]:
# ✏️ STEP 3: 단어 빈도를 계산하고 TOP 5를 출력하세요
# 힌트: Counter() 와 .most_common(5) 를 사용합니다

# Counter로 빈도 계산
영화_빈도 = Counter(영화_단어들)

print("📊 영화 리뷰 TOP 5 키워드")
print()

# 순위별로 출력
for 순위, (단어, 횟수) in enumerate(영화_빈도.most_common(5), 1):
    별 = "★" * 횟수  # 횟수만큼 별 표시
    print(f"  {순위}위: {단어} ({횟수}회) {별}")

📊 영화 리뷰 TOP 5 키워드

  1위: 영화 (3회) ★★★
  2위: 배우 (2회) ★★
  3위: 연기 (2회) ★★
  4위: 최고 (2회) ★★
  5위: 진짜 (1회) ★


In [ ]:
# 🔥 도전! 불용어를 추가해서 결과를 개선해보세요
#
# TOP 5 결과를 보고 분석에 불필요한 단어가 있으면
# 아래 추가_불용어 목록에 넣고 다시 실행해보세요!

# ↓ 여기에 추가하고 싶은 단어를 넣어보세요
추가_불용어 = [
    # 예시: "주의", "생각", "올해"
]

# 기존 불용어 + 새 불용어 합치기
# + 연산자 : 두 리스트를 이어붙입니다
확장_불용어 = 불용어_목록 + 추가_불용어

# 확장된 불용어로 다시 전처리하는 함수
def 전처리_파이프라인_v2(텍스트):
    텍스트 = re.sub(r'http\S+', '', 텍스트)
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)
    텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()
    토큰들 = okt.nouns(텍스트)
    # 확장된 불용어 목록 사용
    결과 = [t for t in 토큰들 if t not in 확장_불용어 and len(t) >= 2]
    return 결과


# v2로 다시 전처리
개선_단어들 = []
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰1))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰2))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰3))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰4))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰5))

개선_빈도 = Counter(개선_단어들)

print("📊 개선 후 TOP 5")
for 순위, (단어, 횟수) in enumerate(개선_빈도.most_common(5), 1):
    print(f"  {순위}위: {단어} ({횟수}회)")

📊 개선 후 TOP 5
  1위: 영화 (3회)
  2위: 배우 (2회)
  3위: 연기 (2회)
  4위: 최고 (2회)
  5위: 진짜 (1회)


---
## 📌 2주차 핵심 정리

| 단계 | 개념 | 핵심 코드 |
|------|------|----------|
| 1️⃣ 정제 | 노이즈 제거 | `re.sub(r'[^가-힣 ]', '', text)` |
| 2️⃣ 정규화 | 표현 통일 | `re.sub(r'(.)\1{2,}', r'\1\1', text)` |
| 3️⃣ 토큰화 | 단어로 쪼개기 | `okt.nouns(text)` |
| 4️⃣ 불용어 | 불필요 단어 제거 | `[w for w in words if w not in 불용어]` |

**KoNLPy 4가지 메서드:**

```python
okt.morphs(문장)             # 형태소로 쪼개기
okt.morphs(문장, stem=True)  # 형태소 + 원형으로 통일
okt.pos(문장)                # 형태소 + 품사 함께 보기
okt.nouns(문장)              # 명사만 뽑기  ← 가장 많이 사용!
```

---

### 🗺️ 다음 주 예고 — 3주차: 텍스트 수치화

```
이번 주 결과:  ["치킨", "배달", "음식", "서비스"]
다음 주 목표:  {"치킨": 0.45, "배달": 0.12, "음식": 0.78, ...}  ← TF-IDF!
```

단어를 숫자로 변환해서 컴퓨터가 이해할 수 있게 만드는 **TF-IDF** 기법을 배웁니다.  
오늘 만든 전처리 파이프라인이 그대로 사용됩니다! 🚀

In [1]:
from collections import Counter
import re

# (참고) 만약 Okt 형태소 분석기를 사용하신다면 아래 주석을 해제하세요.
# from konlpy.tag import Okt
# okt = Okt()

# ── 과제용 영화 리뷰 데이터 (신규 데이터 포함) ──────────────────

리뷰1 = "진짜 뻥안치고 ㅈㄴ 웃겼다 계속 웃으면서 봤어요!"
리뷰2 = "지금까지 이런맛은 없었다 이것은 갈비인가 통닭인가 수원왕갈비통닭입니다."
리뷰3 = "완전 내스타일.. 류승룡 영화 요즘 말아먹더니 이번엔 대박 웃기네 ㅋㅋ 역시 갓승룡"
리뷰4 = "자꾸 광고할떄 치킨치킨 거리길래 뭔말인가 했더만..ㅋㅋㅋ 치킨먹고싶네"
리뷰5 = "올해 가장 재미있었다 아직까진"

print("🎬 신규 영화 리뷰 5개 준비 완료!")
print("-" * 30)

🎬 신규 영화 리뷰 5개 준비 완료!
------------------------------


In [2]:
# ✏️ STEP 1: 각 리뷰를 전처리_파이프라인으로 처리하세요
# (정규표현식을 활용해 특수문자를 제거하는 함수라고 가정합니다)

def 전처리_파이프라인(텍스트):
    # 특수문자 제거 및 공백 정리
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)
    # 단어 단위로 분리 (간단한 토큰화)
    return 텍스트.split()

결과1 = 전처리_파이프라인(리뷰1)
결과2 = 전처리_파이프라인(리뷰2)
결과3 = 전처리_파이프라인(리뷰3)
결과4 = 전처리_파이프라인(리뷰4)
결과5 = 전처리_파이프라인(리뷰5)

print("리뷰1 분석 결과:", 결과1)
print("리뷰2 분석 결과:", 결과2)


리뷰1 분석 결과: ['진짜', '뻥안치고', '웃겼다', '계속', '웃으면서', '봤어요']
리뷰2 분석 결과: ['지금까지', '이런맛은', '없었다', '이것은', '갈비인가', '통닭인가', '수원왕갈비통닭입니다']


In [3]:
# ✏️ STEP 2: 모든 단어를 하나의 리스트로 합치세요 (extend 사용)

모든_단어들 = []

모든_단어들.extend(결과1)
모든_단어들.extend(결과2)
모든_단어들.extend(결과3)
모든_단어들.extend(결과4)
모든_단어들.extend(결과5)

print(f"\n📊 수집된 총 단어 수: {len(모든_단어들)}개")



📊 수집된 총 단어 수: 35개


In [4]:
# ✏️ STEP 3: 단어 빈도를 계산하고 TOP 5를 출력하세요

# 빈도 계산
단어_카운트 = Counter(모든_단어들)

print("\n🏆 영화 리뷰 인기 키워드 TOP 5")
print("-" * 30)

for 순위, (단어, 횟수) in enumerate(단어_카운트.most_common(5), 1):
    별점 = "★" * 횟수
    print(f"  {순위}위: {단어:<10} ({횟수}회) {별점}")



🏆 영화 리뷰 인기 키워드 TOP 5
------------------------------
  1위: 진짜         (1회) ★
  2위: 뻥안치고       (1회) ★
  3위: 웃겼다        (1회) ★
  4위: 계속         (1회) ★
  5위: 웃으면서       (1회) ★


In [5]:
# 🔥 도전! 불용어를 추가해서 의미 없는 단어 제거하기

추가_불용어 = ["진짜", "계속", "요즘", "아직까진"] # 분석에서 뺄 단어들

# 불용어가 제거된 최종 리스트
최종_단어들 = [단어 for 단어 in 모든_단어들 if 단어 not in 추가_불용어]

print("\n✨ 불용어 제거 후 TOP 5 결과")
최종_카운트 = Counter(최종_단어들)
for 순위, (단어, 횟수) in enumerate(최종_카운트.most_common(5), 1):
    print(f"  {순위}위: {단어} ({횟수}회)")


✨ 불용어 제거 후 TOP 5 결과
  1위: 뻥안치고 (1회)
  2위: 웃겼다 (1회)
  3위: 웃으면서 (1회)
  4위: 봤어요 (1회)
  5위: 지금까지 (1회)
